# Inference with scikit-learn and PyTorch

**Author/contact:** Jonathan Funk ([jonfu@dtu.dk](mailto:jonfu@dtu.dk))


In this notebook you will use open Python tools to perform the main steps of a protein design workflow: compute sequence representations, visualize representation spaces, train predictive models, estimate uncertainty with an ensemble, rank candidate sequences, and inspect simple structure-derived residue contacts.

The workflow intentionally uses explicit code instead of a proprietary wrapper. Most modeling is done with `scikit-learn`; the neural-network example uses `torch`.

## Learning goals and discussion format

This notebook is about decisions after a model exists. For each ranking or uncertainty estimate, ask whether it would change which experiment you run next. Discuss the decision, not only the metric.

In [ ]:
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.spatial.distance import cdist
from scipy.stats import kendalltau, pearsonr
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.manifold import TSNE
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Load the enzyme library

The `Nitric_Oxide_Dioxygenase.csv` file contains protein variants, mutation labels, and measured activity values. We keep the data in a normal `pandas` dataframe so the relationship between columns, features, and labels is visible.

In [ ]:
enzyme_data = Path("Nitric_Oxide_Dioxygenase.csv")
df = pd.read_csv(enzyme_data)

seq_col = "Sequence"
name_col = "Description"
y_col = "Data"

print(df.shape)
df.head()

## BLOSUM62 sequence representation

BLOSUM62 maps each amino acid to a 20-dimensional substitution-score vector. A fixed-length protein sequence can therefore be represented by concatenating the per-position BLOSUM62 vectors.

In [ ]:
AMINO_ACIDS = "ARNDCQEGHILKMFPSTWYV"
BLOSUM62 = np.array([
    [ 4,-1,-2,-2, 0,-1,-1, 0,-2,-1,-1,-1,-1,-2,-1, 1, 0,-3,-2, 0],
    [-1, 5, 0,-2,-3, 1, 0,-2, 0,-3,-2, 2,-1,-3,-2,-1,-1,-3,-2,-3],
    [-2, 0, 6, 1,-3, 0, 0, 0, 1,-3,-3, 0,-2,-3,-2, 1, 0,-4,-2,-3],
    [-2,-2, 1, 6,-3, 0, 2,-1,-1,-3,-4,-1,-3,-3,-1, 0,-1,-4,-3,-3],
    [ 0,-3,-3,-3, 9,-3,-4,-3,-3,-1,-1,-3,-1,-2,-3,-1,-1,-2,-2,-1],
    [-1, 1, 0, 0,-3, 5, 2,-2, 0,-3,-2, 1, 0,-3,-1, 0,-1,-2,-1,-2],
    [-1, 0, 0, 2,-4, 2, 5,-2, 0,-3,-3, 1,-2,-3,-1, 0,-1,-3,-2,-2],
    [ 0,-2, 0,-1,-3,-2,-2, 6,-2,-4,-4,-2,-3,-3,-2, 0,-2,-2,-3,-3],
    [-2, 0, 1,-1,-3, 0, 0,-2, 8,-3,-3,-1,-2,-1,-2,-1,-2,-2, 2,-3],
    [-1,-3,-3,-3,-1,-3,-3,-4,-3, 4, 2,-3, 1, 0,-3,-2,-1,-3,-1, 3],
    [-1,-2,-3,-4,-1,-2,-3,-4,-3, 2, 4,-2, 2, 0,-3,-2,-1,-2,-1, 1],
    [-1, 2, 0,-1,-3, 1, 1,-2,-1,-3,-2, 5,-1,-3,-1, 0,-1,-3,-2,-2],
    [-1,-1,-2,-3,-1, 0,-2,-3,-2, 1, 2,-1, 5, 0,-2,-1,-1,-1,-1, 1],
    [-2,-3,-3,-3,-2,-3,-3,-3,-1, 0, 0,-3, 0, 6,-4,-2,-2, 1, 3,-1],
    [-1,-2,-2,-1,-3,-1,-1,-2,-2,-3,-3,-1,-2,-4, 7,-1,-1,-4,-3,-2],
    [ 1,-1, 1, 0,-1, 0, 0, 0,-1,-2,-2, 0,-1,-2,-1, 4, 1,-3,-2,-2],
    [ 0,-1, 0,-1,-1,-1,-1,-2,-2,-1,-1,-1,-1,-2,-1, 1, 5,-2,-2, 0],
    [-3,-3,-4,-4,-2,-2,-3,-2,-2,-3,-2,-3,-1, 1,-4,-3,-2,11, 2,-3],
    [-2,-2,-2,-3,-2,-1,-2,-3, 2,-1,-1,-2,-1, 3,-3,-2,-2, 2, 7,-1],
    [ 0,-3,-3,-3,-1,-2,-2,-3,-3, 3, 1,-2, 1,-1,-2,-2, 0,-3,-1, 4],
], dtype=np.float32)
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}


def blosum_encode(sequence):
    rows = []
    for aa in sequence:
        rows.append(BLOSUM62[AA_TO_INDEX.get(aa, AA_TO_INDEX["A"])])
    return np.asarray(rows, dtype=np.float32).ravel()


lengths = df[seq_col].str.len()
print(lengths.describe())
assert lengths.nunique() == 1, "This simple representation expects aligned fixed-length sequences."

X_blosum = np.vstack(df[seq_col].map(blosum_encode))
y = df[y_col].to_numpy(dtype=np.float32)
X_blosum.shape

## Visualize representation space

Dimensionality reduction helps reveal whether variants with similar measured activity cluster together. PCA is deterministic and fast; t-SNE often separates local neighborhoods more clearly.

### Visualization caution

Before interpreting clusters, explain to a neighbor why t-SNE distances between far-away points can be misleading. What claims can you safely make from the plot, and what claims require a quantitative test?

In [ ]:
X_scaled = StandardScaler().fit_transform(X_blosum)

pca_xy = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled)
tsne_xy = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, xy, title in zip(axes, [pca_xy, tsne_xy], ["PCA", "t-SNE"]):
    scatter = ax.scatter(xy[:, 0], xy[:, 1], c=y, cmap="viridis", s=24, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("component 1")
    ax.set_ylabel("component 2")
fig.colorbar(scatter, ax=axes, label=y_col)
plt.tight_layout()

## Train a random forest regressor

A random forest is a strong baseline for small protein datasets. We use a held-out test set for final evaluation and cross-validation on the training set to estimate typical model performance.

In [ ]:
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X_blosum, y, np.arange(len(df)), test_size=0.2, random_state=SEED
)

rf = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"Test R-squared: {r2_score(y_test, y_pred):.3f}")
print(f"Test MAE:       {mean_absolute_error(y_test, y_pred):.3f}")
print(f"Pearson r:      {pearsonr(y_test, y_pred).statistic:.3f}")
print(f"Kendall tau:    {kendalltau(y_test, y_pred).statistic:.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, y_pred, alpha=0.8)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "k--", lw=1)
ax.set_xlabel("Measured activity")
ax.set_ylabel("Predicted activity")
ax.set_title("Random forest validation")

## Cross-validation and uncertainty from an ensemble

Proteins are usually expensive to test, so model uncertainty matters. Here we train one model per cross-validation split. The mean prediction is the ensemble estimate; the standard deviation across models is a simple uncertainty proxy.

### Decision checkpoint

If high uncertainty correlates poorly with true error, should you still use it for experiment selection? Discuss when uncertainty is useful for exploration even if it is not perfectly calibrated.

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)
oof_pred = np.zeros_like(y, dtype=np.float32)
ensemble = []

for fold, (tr, val) in enumerate(kf.split(X_blosum), start=1):
    model = RandomForestRegressor(n_estimators=200, random_state=SEED + fold, n_jobs=-1)
    model.fit(X_blosum[tr], y[tr])
    oof_pred[val] = model.predict(X_blosum[val])
    ensemble.append(model)

residual = np.abs(y - oof_pred)
ensemble_preds = np.vstack([model.predict(X_blosum) for model in ensemble])
uncertainty = ensemble_preds.std(axis=0)

print(f"CV R-squared: {r2_score(y, oof_pred):.3f}")
print(f"CV MAE:       {mean_absolute_error(y, oof_pred):.3f}")
print(f"Error/uncertainty Pearson r: {pearsonr(residual, uncertainty).statistic:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(uncertainty, residual, alpha=0.7)
ax.set_xlabel("Ensemble prediction std")
ax.set_ylabel("Absolute prediction error")
ax.set_title("Does uncertainty track error?")

**Task**: Try the same uncertainty analysis with a different representation or model. Does uncertainty become more informative?

## A small PyTorch regressor

The Torch example below trains a compact neural network on the same BLOSUM features. This is not meant to beat the random forest; it shows the minimal pieces needed for a custom differentiable model.

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


scaler = StandardScaler().fit(X_train)
X_train_t = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
torch_model = MLPRegressor(X_train.shape[1])
optimizer = torch.optim.AdamW(torch_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

for epoch in range(100):
    torch_model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(torch_model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch + 1:3d} | train MSE {epoch_loss / len(loader.dataset):.4f}")

torch_model.eval()
with torch.no_grad():
    torch_pred = torch_model(X_test_t).numpy()

print(f"Torch test R-squared: {r2_score(y_test, torch_pred):.3f}")
print(f"Torch test MAE:       {mean_absolute_error(y_test, torch_pred):.3f}")

## Rank candidate sequences with an acquisition score

A simple upper confidence bound score balances predicted activity and uncertainty. In a real design campaign, candidate generation should respect biochemical and experimental constraints. Here we demonstrate the ranking step by sampling single mutants around the best observed variant.

### Design meeting prompt

Imagine you can test only 10 variants. One person argues for exploitation, one argues for exploration, and one argues for diversity or feasibility constraints. Decide together how you would modify the acquisition score.

In [ ]:
def single_mutants(sequence, positions=None, alphabet=AMINO_ACIDS):
    positions = range(len(sequence)) if positions is None else positions
    for pos in positions:
        for aa in alphabet:
            if aa != sequence[pos]:
                yield sequence[:pos] + aa + sequence[pos + 1:], pos + 1, aa


best_row = df.iloc[df[y_col].idxmax()]
# Keep this small for class runtime. Increase or replace with a designed candidate pool as needed.
positions_to_scan = range(0, len(best_row[seq_col]), 10)
candidate_rows = []
for seq, position, aa in single_mutants(best_row[seq_col], positions_to_scan):
    original = best_row[seq_col][position - 1]
    candidate_rows.append((seq, f"{original}{position}{aa}"))

candidates = pd.DataFrame(candidate_rows, columns=[seq_col, "Mutation"])
X_candidates = np.vstack(candidates[seq_col].map(blosum_encode))
candidate_preds = np.vstack([model.predict(X_candidates) for model in ensemble])

candidates["predicted_activity"] = candidate_preds.mean(axis=0)
candidates["uncertainty"] = candidate_preds.std(axis=0)
candidates["ucb_score"] = candidates["predicted_activity"] + 0.1 * candidates["uncertainty"]

candidates.sort_values("ucb_score", ascending=False).head(10)

**Optional Task**: Extend the ranking to multiple objectives. One practical route is to train one model per objective, normalize each prediction, and keep candidates on the Pareto front.

# Structure-derived features with NumPy

The PDB file can be parsed directly to extract chain sequences, residue coordinates, and residues close to non-protein atoms. This is a lightweight alternative to a structure wrapper and is enough for simple contact-based design discussions.

### Structure reasoning prompt

Why might residues near a ligand be protected from mutation in one design campaign but targeted in another? Discuss how the answer depends on whether you want to preserve binding, change specificity, or improve stability.

In [ ]:
THREE_TO_ONE = {
    "ALA":"A", "ARG":"R", "ASN":"N", "ASP":"D", "CYS":"C", "GLN":"Q", "GLU":"E", "GLY":"G",
    "HIS":"H", "ILE":"I", "LEU":"L", "LYS":"K", "MET":"M", "PHE":"F", "PRO":"P", "SER":"S",
    "THR":"T", "TRP":"W", "TYR":"Y", "VAL":"V",
}


def parse_pdb(path):
    residues = {}
    ligand_atoms = []
    with open(path) as handle:
        for line in handle:
            record = line[:6].strip()
            if record not in {"ATOM", "HETATM"}:
                continue
            atom = line[12:16].strip()
            resname = line[17:20].strip()
            chain = line[21].strip()
            resseq = int(line[22:26])
            coord = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            if record == "ATOM" and resname in THREE_TO_ONE:
                key = (chain, resseq, resname)
                residues.setdefault(key, {})[atom] = coord
            elif record == "HETATM" and resname not in {"HOH", "WAT"}:
                ligand_atoms.append((resname, chain, resseq, atom, coord))
    return residues, ligand_atoms


residues, ligand_atoms = parse_pdb("1zb6.pdb")
chain_a = [(key, atoms) for key, atoms in residues.items() if key[0] == "A"]
sequence_a = "".join(THREE_TO_ONE[key[2]] for key, atoms in chain_a if "CA" in atoms)

print(f"Chain A residues with CA atoms: {len(sequence_a)}")
print(sequence_a[:80] + "...")
print(f"Non-water ligand atoms: {len(ligand_atoms)}")

In [ ]:
def residues_near_ligand(residues, ligand_atoms, chain="A", distance_cutoff=7.0):
    ligand_xyz = np.vstack([atom[-1] for atom in ligand_atoms])
    contacts = []
    for (res_chain, resseq, resname), atoms in residues.items():
        if res_chain != chain or "CA" not in atoms:
            continue
        atom_xyz = np.vstack(list(atoms.values()))
        min_dist = cdist(atom_xyz, ligand_xyz).min()
        if min_dist <= distance_cutoff:
            contacts.append({"chain": res_chain, "residue_number": resseq, "residue_name": resname, "min_distance": min_dist})
    return pd.DataFrame(contacts).sort_values("min_distance")


interface = residues_near_ligand(residues, ligand_atoms, chain="A", distance_cutoff=7.0)
interface.head(20)

## Contact-preserving mutation suggestions

Without inverse folding software, we can still express design constraints explicitly: keep residues near the ligand fixed and mutate only positions outside the contact set. The example below lists random allowed positions as a starting point for a candidate library.

In [ ]:
rng = np.random.default_rng(SEED)
fixed_positions = set(interface["residue_number"])
mutable_positions = [resseq for (chain, resseq, resname), atoms in residues.items() if chain == "A" and resseq not in fixed_positions and "CA" in atoms]

suggestions = []
for resseq in rng.choice(mutable_positions, size=10, replace=False):
    current = next(THREE_TO_ONE[key[2]] for key, atoms in residues.items() if key[0] == "A" and key[1] == resseq)
    choices = [aa for aa in AMINO_ACIDS if aa != current]
    suggestions.append({"position": int(resseq), "current": current, "suggested": rng.choice(choices)})

pd.DataFrame(suggestions).sort_values("position")